# Synthetic Text Diversity Evaluation

This notebook evaluates lexical diversity of synthetic data using metrics from Texygen and related benchmarks:

1. **Train–test overlap detection** – MinHash over word shingles to flag train sentences that are exact or >90% similar to test data. Matches are reported and removed before metrics. Internal near-duplicates within train are **not** checked.
2. **Distinct-n** – Fraction of unique n-grams in the corpus. Higher values indicate more diverse text; low values suggest mode collapse or repetition.
3. **Self-BLEU** – BLEU of each synthetic sentence against the rest of the corpus. Lower values indicate more diverse generations; high Self-BLEU warns of repetitive or low-entropy text.

## 1. Setup

In [1]:
%pip install -q pandas nltk


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib
import os
import struct
from pathlib import Path
from typing import Dict, List, Set, Tuple

import pandas as pd
import nltk
from IPython.display import display

nltk.download("punkt_tab", quiet=True)

DATA_DIR = Path("data/processed")
SYNTHETIC_PATH = DATA_DIR / "train_1500_gen_eval.csv"
TEST_PATH = DATA_DIR / "test_final.csv"
ORIGINAL_PATH = DATA_DIR / "train_org_processed.csv"
DEDUP_OUTPUT_PATH = DATA_DIR / "train_1500_gen_eval_dedup.csv"
SIMILARITY_THRESHOLD = 0.90
SHINGLE_SIZE = 3
NUM_PERM = 128

## 2. Diversity Metrics

In [3]:
def tokenize(text: str) -> List[str]:
    """Tokenize text (whitespace split). Swap for language-specific tokenizer if needed."""
    return str(text).strip().split() if pd.notna(text) and str(text).strip() else []


def get_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    """Extract n-grams from token list."""
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)] if len(tokens) >= n else []


def compute_distinct_n(texts: List[str], n: int = 2) -> dict:
    """
    Compute Distinct-n: fraction of unique n-grams in the corpus.
    Higher = more diverse. Low values suggest mode collapse or repetition.

    distinct_n = |unique n-grams| / |total n-grams|
    """
    all_ngrams = []
    for text in texts:
        tokens = tokenize(text)
        all_ngrams.extend(get_ngrams(tokens, n))

    total = len(all_ngrams)
    unique = len(set(all_ngrams))
    distinct = unique / total if total > 0 else 0.0

    return {"unique_ngrams": unique, "total_ngrams": total, f"distinct_{n}": distinct}

In [4]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def compute_self_bleu(texts: List[str], max_n: int = 4, sample_size: int = None) -> dict:
    """
    Compute Self-BLEU: BLEU of each sentence against the rest of the corpus.
    Lower = more diverse. High Self-BLEU warns of repetitive or low-entropy text.

    Uses geometric mean of BLEU-1 to BLEU-4 (Texygen-style).
    """
    tokenized = [tokenize(t) for t in texts if tokenize(t)]
    if len(tokenized) < 2:
        return {"self_bleu": 0.0, "n_sentences": len(tokenized)}

    if sample_size and len(tokenized) > sample_size:
        import random
        tokenized = random.sample(tokenized, sample_size)

    smoothing = SmoothingFunction().method1
    scores = []

    for i, hyp in enumerate(tokenized):
        refs = [tok for j, tok in enumerate(tokenized) if j != i]
        if not refs or not hyp:
            continue
        bleu_n = []
        for n in range(1, max_n + 1):
            if len(hyp) >= n:
                w = [1.0 / n] * n
                s = sentence_bleu(refs, hyp, weights=tuple(w), smoothing_function=smoothing)
                bleu_n.append(s)
        if bleu_n:
            geo_mean = 1.0
            for s in bleu_n:
                geo_mean *= s
            geo_mean **= 1.0 / len(bleu_n)
            scores.append(geo_mean)

    avg = sum(scores) / len(scores) if scores else 0.0
    return {"self_bleu": avg, "n_sentences": len(scores)}

## 3. Train–Test Overlap Detection (MinHash)

In [5]:
class MinHash:
    """Lightweight MinHash (stdlib only; no datasketch dependency)."""

    def __init__(self, num_perm: int = NUM_PERM, seed: int = 1):
        self.num_perm = num_perm
        self._max = (1 << 32) - 1
        self.hashes = [self._max] * num_perm
        self._perms = []
        for i in range(num_perm):
            digest = hashlib.md5(f"{seed}_{i}".encode()).digest()
            a, b = struct.unpack("II", digest[:8])
            self._perms.append((a | 1, b))

    def update(self, data: bytes) -> None:
        base = int.from_bytes(hashlib.md5(data).digest()[:4], "little")
        for i, (a, b) in enumerate(self._perms):
            h = (a * base + b) & self._max
            if h < self.hashes[i]:
                self.hashes[i] = h

    def jaccard(self, other: "MinHash") -> float:
        return sum(x == y for x, y in zip(self.hashes, other.hashes)) / self.num_perm


class MinHashLSH:
    """Band-based LSH index for MinHash signatures."""

    def __init__(self, threshold: float, num_perm: int):
        if not 0 < threshold < 1:
            raise ValueError("threshold must be between 0 and 1")
        best = (32, 1)
        best_diff = float("inf")
        for bands in range(1, num_perm + 1):
            if num_perm % bands != 0:
                continue
            rows = num_perm // bands
            est = (1 / bands) ** (1 / rows)
            diff = abs(est - threshold)
            if diff < best_diff:
                best_diff, best = diff, (bands, rows)
        self.bands, self.rows = best
        self.num_perm = num_perm
        self._tables: List[Dict[int, List[str]]] = [dict() for _ in range(self.bands)]

    def _band_keys(self, sig: MinHash) -> List[int]:
        keys = []
        for b in range(self.bands):
            start = b * self.rows
            chunk = sig.hashes[start : start + self.rows]
            keys.append(hash(tuple(chunk)) & 0xFFFFFFFFFFFFFFFF)
        return keys

    def insert(self, key: str, sig: MinHash) -> None:
        for table, band_key in zip(self._tables, self._band_keys(sig)):
            table.setdefault(band_key, []).append(key)

    def query(self, sig: MinHash) -> List[str]:
        candidates: Set[str] = set()
        for table, band_key in zip(self._tables, self._band_keys(sig)):
            candidates.update(table.get(band_key, []))
        return list(candidates)


def normalize_text(text: str) -> str:
    """Lowercase and collapse whitespace for consistent hashing."""
    return " ".join(str(text).lower().split()) if pd.notna(text) else ""


def text_shingles(text: str, n: int = SHINGLE_SIZE) -> List[str]:
    tokens = normalize_text(text).split()
    if len(tokens) < n:
        return [" ".join(tokens)] if tokens else []
    return [" ".join(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]


def build_minhash(text: str, num_perm: int = NUM_PERM, n: int = SHINGLE_SIZE) -> MinHash:
    m = MinHash(num_perm=num_perm)
    for shingle in text_shingles(text, n=n):
        m.update(shingle.encode("utf-8"))
    return m


def jaccard_similarity(sig_a: MinHash, sig_b: MinHash) -> float:
    return sig_a.jaccard(sig_b)


def find_reference_matches(
    query_texts: List[str],
    reference_texts: List[str],
    threshold: float = SIMILARITY_THRESHOLD,
    num_perm: int = NUM_PERM,
) -> List[Tuple[int, int, float]]:
    """Return (query_idx, reference_idx, similarity) for query rows matching reference."""
    ref_signatures = [build_minhash(t, num_perm=num_perm) for t in reference_texts]
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

    for idx, sig in enumerate(ref_signatures):
        lsh.insert(f"r{idx}", sig)

    matches: Dict[int, Tuple[int, float]] = {}
    for q_idx, text in enumerate(query_texts):
        sig = build_minhash(text, num_perm=num_perm)
        candidates = lsh.query(sig)
        if not candidates:
            continue
        best_ref, best_sim = -1, 0.0
        for key in candidates:
            r_idx = int(key[1:])
            sim = jaccard_similarity(sig, ref_signatures[r_idx])
            if sim > best_sim:
                best_ref, best_sim = r_idx, sim
        if best_sim >= threshold:
            matches[q_idx] = (best_ref, best_sim)

    return sorted(
        [(q_idx, ref_idx, sim) for q_idx, (ref_idx, sim) in matches.items()],
        key=lambda x: (-x[2], x[0]),
    )

## 4. Load Data

In [6]:
def resolve_synthetic_path(path: Path) -> Path:
    if path.is_file():
        return path
    alt = path.with_suffix(".csv")
    if alt.is_file():
        return alt
    if path.is_dir():
        csv_files = sorted(path.glob("*.csv"))
        if len(csv_files) == 1:
            return csv_files[0]
        if csv_files:
            raise FileNotFoundError(
                f"Multiple CSV files in '{path}': {[p.name for p in csv_files]}. "
                "Point SYNTHETIC_PATH to a single file."
            )
    raise FileNotFoundError(
        f"Could not find synthetic data at '{path}' or '{path.with_suffix('.csv')}'.\n"
        "Please ensure the file exists and the path is correct."
    )


resolved_synthetic_path = resolve_synthetic_path(SYNTHETIC_PATH)
synthetic_df_raw = pd.read_csv(resolved_synthetic_path)
print(f"Loaded: {resolved_synthetic_path}")
text_column = "Sentence_clean" if "Sentence_clean" in synthetic_df_raw.columns else "Sentence"
print(f"Synthetic dataset (raw): {len(synthetic_df_raw):,} rows")
print(f"Text column: {text_column}")
print(f"Test reference: {TEST_PATH} (exists={TEST_PATH.is_file()})")

Loaded: data/processed/train_1500_gen_eval.csv
Synthetic dataset (raw): 10,558 rows
Text column: Sentence
Test reference: data/processed/test_final.csv (exists=True)


In [7]:
def text_hash(text: str) -> str:
    return hashlib.md5(normalize_text(text).encode("utf-8")).hexdigest()


valid_mask = synthetic_df_raw[text_column].notna() & synthetic_df_raw[text_column].astype(str).str.strip().ne("")
valid_texts = synthetic_df_raw.loc[valid_mask, text_column].astype(str).tolist()
valid_indices = synthetic_df_raw.index[valid_mask].tolist()

# --- Train vs test overlap ---
exact_test_match_indices: Set[int] = set()
near_test_match_indices: Set[int] = set()
test_matches: List[Tuple[int, int, float]] = []

if not TEST_PATH.is_file():
    raise FileNotFoundError(f"Test file not found: {TEST_PATH}")

test_df = pd.read_csv(TEST_PATH)
test_text_col = "Sentence_clean" if "Sentence_clean" in test_df.columns else "Sentence"
test_texts = test_df[test_text_col].dropna().astype(str).tolist()
test_hash_set = {text_hash(t) for t in test_texts if normalize_text(t)}

print(f"Checking {len(valid_texts):,} train vs {len(test_texts):,} test sentences...")

for row_idx, text in zip(valid_indices, valid_texts):
    if normalize_text(text) and text_hash(text) in test_hash_set:
        exact_test_match_indices.add(row_idx)

print(f"Exact train–test matches: {len(exact_test_match_indices)}")

print(f"Building MinHash signatures for near-match scan (threshold >= {SIMILARITY_THRESHOLD:.0%})...")
test_matches = find_reference_matches(valid_texts, test_texts, threshold=SIMILARITY_THRESHOLD)
near_test_match_indices = {valid_indices[q_idx] for q_idx, _, _ in test_matches}
print(f"Near train–test matches (>={SIMILARITY_THRESHOLD:.0%}): {len(near_test_match_indices)}")

if test_matches:
    print("\nExamples of train ↔ test near-matches:")
    for q_idx, t_idx, sim in test_matches[:5]:
        print(f"  sim={sim:.3f} | train[{valid_indices[q_idx]}] {valid_texts[q_idx][:70]}…")
        print(f"           | test [{t_idx}] {test_texts[t_idx][:70]}…")

# --- Remove train rows that overlap test ---
overlap_indices = exact_test_match_indices | near_test_match_indices
remove_mask = synthetic_df_raw.index.to_series().isin(overlap_indices)
n_remove = int(remove_mask.sum())

near_only_indices = near_test_match_indices - exact_test_match_indices

match_report = pd.DataFrame(
    {
        "category": ["exact_train_test_match", "near_train_test_match_only", "total_removed"],
        "count": [
            len(exact_test_match_indices),
            len(near_only_indices),
            n_remove,
        ],
    }
)
print("\n=== Train–Test Overlap Report ===")
display(match_report)

synthetic_df = synthetic_df_raw.loc[~remove_mask].reset_index(drop=True)
synthetic_texts = synthetic_df[text_column].dropna().astype(str).tolist()

DATA_DIR.mkdir(parents=True, exist_ok=True)
synthetic_df.to_csv(DEDUP_OUTPUT_PATH, index=False)
print(f"\nRemoved {n_remove:,} train rows overlapping test → {len(synthetic_df):,} remaining")
print(f"Cleaned train data saved to: {DEDUP_OUTPUT_PATH}")

Checking 10,558 train vs 692 test sentences...
Exact train–test matches: 0
Building MinHash signatures for near-match scan (threshold >= 90%)...
Near train–test matches (>=90%): 0

=== Train–Test Overlap Report ===


,category,count
0,exact_train_test_match,0
1,near_train_test_match_only,0
2,total_removed,0



Removed 0 train rows overlapping test → 10,558 remaining
Cleaned train data saved to: data/processed/train_1500_gen_eval_dedup.csv


## 5. Compute Diversity Metrics

In [8]:
# Distinct-n (n=1,2,3)
print("Computing Distinct-n...")
distinct_1 = compute_distinct_n(synthetic_texts, n=1)
distinct_2 = compute_distinct_n(synthetic_texts, n=2)
distinct_3 = compute_distinct_n(synthetic_texts, n=3)

print("\n=== Distinct-n ===")
print(f"Distinct-1: {distinct_1['distinct_1']:.4f} (unique unigrams: {distinct_1['unique_ngrams']:,} / {distinct_1['total_ngrams']:,})")
print(f"Distinct-2: {distinct_2['distinct_2']:.4f} (unique bigrams: {distinct_2['unique_ngrams']:,} / {distinct_2['total_ngrams']:,})")
print(f"Distinct-3: {distinct_3['distinct_3']:.4f} (unique trigrams: {distinct_3['unique_ngrams']:,} / {distinct_3['total_ngrams']:,})")

Computing Distinct-n...

=== Distinct-n ===
Distinct-1: 0.0471 (unique unigrams: 6,524 / 138,476)
Distinct-2: 0.4508 (unique bigrams: 57,671 / 127,918)
Distinct-3: 0.7860 (unique trigrams: 92,247 / 117,366)


## 6. Optional: Compare with Original Data

In [9]:
if ORIGINAL_PATH.is_file():
    if "original_df" not in globals():
        original_df = pd.read_csv(ORIGINAL_PATH)
    orig_text_col = "Sentence_clean" if "Sentence_clean" in original_df.columns else "Sentence"
    original_texts = original_df[orig_text_col].dropna().astype(str).tolist()
    print(f"Original dataset: {len(original_texts)} samples (deduplicated synthetic: {len(synthetic_texts)})")

    orig_d1 = compute_distinct_n(original_texts, n=1)
    orig_d2 = compute_distinct_n(original_texts, n=2)
    orig_d3 = compute_distinct_n(original_texts, n=3)

    summary = pd.DataFrame({
        "Metric": ["Distinct-1", "Distinct-2", "Distinct-3"],
        "Original": [orig_d1["distinct_1"], orig_d2["distinct_2"], orig_d3["distinct_3"]],
        "Synthetic": [distinct_1["distinct_1"], distinct_2["distinct_2"], distinct_3["distinct_3"]],
    })
    summary["Synthetic - Original"] = summary["Synthetic"] - summary["Original"]
    display(summary)
else:
    print("Original file not found. Skipping comparison.")

Original dataset: 5548 samples (deduplicated synthetic: 10558)


,Metric,Original,Synthetic,Synthetic - Original
0,Distinct-1,0.059729,0.047113,-0.012616
1,Distinct-2,0.593488,0.450844,-0.142644
2,Distinct-3,0.930389,0.785977,-0.144411
